<a href="https://colab.research.google.com/github/LBDillon/decoding-design-bias/blob/main/score_proteins_esm3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESM3-open (1.4B) scoring of the experimental-PDB cohort

Scores the 876 PDB chains in `cohort_pdb_scoring_inputs.csv` in two modes. Both use
masked pseudo-log-likelihood: each residue is masked in turn and its native
log-probability is read. The output has the two ESM3 score columns of
`data/pdb/cohort_pdb_scored.csv`.

| Score | Structure track | Scoring context |
| --- | --- | --- |
| `esm3_struct_cond_score` | VQ-VAE tokens from the PDB chain | structure + native sequence |
| `esm3_seq_only_score`    | masked (no structure)            | sequence-only |

The two modes share architecture, tokenizer and scoring; only structure conditioning changes.

Chains longer than 1,500 residues are skipped.

## 1. Setup: GPU, deps, HF auth

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q esm biopython requests tqdm

In [ ]:
# Accept the gated repo at https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1 first.
from huggingface_hub import login
login()

## 2. Mount Drive and set paths

The output CSV is written to Drive so that it survives a runtime disconnect.
Upload `cohort_pdb_scoring_inputs.csv` and `cohort_pdb_scoring_bundle.zip` (the
PDB chain files) to `/content/` first.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os

DATASET = '/content/cohort_pdb_scoring_inputs.csv'

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding_bias_results/PDB/esm3'
OUTPUT        = f'{DRIVE_OUT_DIR}/esm3_pdb_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)

In [ ]:
!unzip /content/cohort_pdb_scoring_bundle.zip -d /content/

## 3. Load model (≈ 1 min; 5GB download first time)

In [ ]:
import torch
from esm.models.esm3 import ESM3

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

model = ESM3.from_pretrained('esm3-open').to(device).eval()
tokenizers = model.tokenizers
mask_id = tokenizers.sequence.mask_token_id
print('mask_id =', mask_id, ' | param count:', sum(p.numel() for p in model.parameters())/1e9, 'B')

## 4. Structure lookup and scoring functions

In [ ]:
import os

PDB_FILES_LOCATION = "/content/cohort_chain_structs"


def fetch_pdb(row):
    """Path of the extracted chain file named in the row's chain_pdb_path, or None."""
    if 'chain_pdb_path' not in row:
        return None
    local_path = os.path.join(PDB_FILES_LOCATION, os.path.basename(row['chain_pdb_path']))
    return local_path if os.path.exists(local_path) else None

In [ ]:
import torch.nn.functional as F
from esm.sdk.api import ESMProtein
from esm.utils.structure.protein_chain import ProteinChain
@torch.no_grad()
def score_protein(pdb_path, max_tokens_per_batch=2048):
    chain = ProteinChain.from_pdb(pdb_path)
    if not chain.sequence:
        return None

    protein = ESMProtein.from_protein_chain(chain)
    pt = model.encode(protein)

    seq_tokens = pt.sequence.to(device)
    struct_tokens = pt.structure.to(device) if pt.structure is not None else None
    L = seq_tokens.shape[0] - 2
    native = seq_tokens[1:1 + L]

    def _run(batch_size):
        results = {}
        for mode_name, use_struct in [('struct_cond', True), ('seq_only', False)]:
            lps = torch.zeros(L, device=device)
            for start in range(0, L, batch_size):
                end = min(start + batch_size, L)
                B = end - start
                seq_batch = seq_tokens.unsqueeze(0).repeat(B, 1).clone()
                for j in range(B):
                    seq_batch[j, start + j + 1] = mask_id
                struct_batch = (struct_tokens.unsqueeze(0).repeat(B, 1)
                                if use_struct and struct_tokens is not None else None)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    esmout = model.forward(sequence_tokens=seq_batch,
                                           structure_tokens=struct_batch)
                logits = esmout.sequence_logits.float()
                for j in range(B):
                    pos = start + j + 1
                    lp = F.log_softmax(logits[j, pos], dim=-1)
                    lps[start + j] = lp[native[start + j]]
                del esmout, logits
            results[f'{mode_name}_mean'] = lps.mean().item()
            results[f'{mode_name}_sum']  = lps.sum().item()
        return results

    batch_size = max(1, max_tokens_per_batch // (L + 2))
    while True:
        try:
            out = _run(batch_size)
            out['length'] = L
            return out
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f'  OOM, retrying L={L} batch={batch_size}')

## 5. Check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    t0 = time.time()
    pdb = fetch_pdb(r)
    t_dl = time.time() - t0
    if pdb is None:
        print(entry, 'no PDB'); continue
    t0 = time.time()
    res = score_protein(pdb)
    t_sc = time.time() - t0
    print(f'{entry} L={res["length"]} struct_cond={res["struct_cond_mean"]:.4f} seq_only={res["seq_only_mean"]:.4f}  dl={t_dl:.1f}s score={t_sc:.1f}s')

## 5. Full run

Writes a fresh `OUTPUT` file and flushes it after every protein.

In [ ]:
from tqdm.auto import tqdm

todo = [
    r for r in rows
    if int(float(r.get('sequence_length') or len(r.get('sequence', '')))) <= 1500
]
print('to score:', len(todo), 'proteins of at most 1,500 residues')

with open(OUTPUT, 'w', newline='') as out:
    w = csv.writer(out)
    w.writerow([
        'Entry', 'species', 'domain',
        'esm3_struct_cond_score', 'esm3_seq_only_score',
        'esm3_struct_cond_sum', 'esm3_seq_only_sum',
        'scored_length', 'dataset_length',
    ])

    counts = {'ok': 0, 'missing_pdb': 0, 'error': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='esm3'):
        entry = r['Entry']
        pdb = fetch_pdb(r)
        if pdb is None:
            counts['missing_pdb'] += 1
            continue
        try:
            res = score_protein(pdb)
        except Exception as exc:
            print(f'[{entry}] {exc}')
            counts['error'] += 1
            continue
        if res is None:
            counts['error'] += 1
            continue
        w.writerow([
            entry, r.get('species', ''), r.get('domain', ''),
            f"{res['struct_cond_mean']:.6f}", f"{res['seq_only_mean']:.6f}",
            f"{res['struct_cond_sum']:.6f}", f"{res['seq_only_sum']:.6f}",
            res['length'], len(r.get('sequence', '')),
        ])
        out.flush()
        torch.cuda.empty_cache()
        counts['ok'] += 1

print('done in', round(time.time() - t_start, 1), 's', counts)

## 7. Summary

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
df[['esm3_struct_cond_score','esm3_seq_only_score']].describe()